In [8]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()
CSV_PATH = os.path.join(BASE_DIR, "car_data_augmented.csv")
FIG = os.path.join(BASE_DIR, "figs_t1") + os.sep
os.makedirs(FIG, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print("Raw shape:", df.shape)
print(df.head())
print(df.info())

print("\n--- Missing values ---")
print(df.isnull().sum())
print("\n--- Duplicates ---", df.duplicated().sum())
print("\n--- Describe ---")
print(df.describe())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["Selling_Price"], kde=True, ax=ax[0], color="#2c6e91")
ax[0].set_title("Distribution of Selling Price (lakhs)")
sns.boxplot(x=df["Selling_Price"], ax=ax[1], color="#2c6e91")
ax[1].set_title("Selling Price - outlier check")
plt.tight_layout()
plt.savefig(FIG + "01_target_distribution.png")
plt.close()

num_cols = ["Selling_Price", "Present_Price", "Kms_Driven", "Year", "Owner"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for a, c in zip(axes.flat, num_cols):
    sns.histplot(df[c], kde=True, ax=a, color="#3b7d4f")
    a.set_title(c)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.savefig(FIG + "02_numeric_distributions.png")
plt.close()

cat_cols = ["Fuel_Type", "Seller_Type", "Transmission"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for a, c in zip(axes, cat_cols):
    sns.countplot(x=df[c], ax=a, palette="viridis", hue=df[c], legend=False)
    a.set_title(c)
plt.tight_layout()
plt.savefig(FIG + "03_categorical_counts.png")
plt.close()

plt.figure(figsize=(6, 5))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix (numeric features)")
plt.tight_layout()
plt.savefig(FIG + "04_correlation_heatmap.png")
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.scatterplot(data=df, x="Present_Price", y="Selling_Price", hue="Fuel_Type", ax=axes[0])
axes[0].set_title("Selling Price vs Present Price")
sns.scatterplot(data=df, x="Kms_Driven", y="Selling_Price", hue="Transmission", ax=axes[1])
axes[1].set_title("Selling Price vs Kms Driven")
sns.scatterplot(data=df, x="Year", y="Selling_Price", hue="Seller_Type", ax=axes[2])
axes[2].set_title("Selling Price vs Year")
plt.tight_layout()
plt.savefig(FIG + "05_scatter_relationships.png")
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for a, c in zip(axes, cat_cols):
    sns.boxplot(data=df, x=c, y="Selling_Price", ax=a, palette="Set2", hue=df[c], legend=False)
    a.set_title(f"Selling Price by {c}")
plt.tight_layout()
plt.savefig(FIG + "06_price_by_category.png")
plt.close()

data = df.copy()

before = len(data)
data = data.drop_duplicates()
print(f"\nDropped {before - len(data)} duplicate rows")

reference_year = data["Year"].max() + 1
data["Car_Age"] = reference_year - data["Year"]
data = data.drop(columns=["Year"])

data = data.drop(columns=["Car_Name"])

data = pd.get_dummies(data, columns=["Fuel_Type", "Seller_Type", "Transmission"],
                       drop_first=True)

bool_cols = data.select_dtypes("bool").columns
data[bool_cols] = data[bool_cols].astype(int)

print("\nWrangled columns:", list(data.columns))
print("Wrangled shape:", data.shape)

X = data.drop(columns=["Selling_Price"])
y = data["Selling_Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

coef_table = pd.DataFrame({"Feature": X.columns, "Coefficient": model.coef_})
coef_table = coef_table.sort_values("Coefficient", key=abs, ascending=False)
print("\nIntercept:", model.intercept_)
print(coef_table)

def metrics(y_true, y_pred, n, k):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return dict(MAE=mae, MSE=mse, RMSE=rmse, R2=r2, Adj_R2=adj_r2, MAPE=mape)

n_feat = X.shape[1]
train_metrics = metrics(y_train, y_train_pred, len(y_train), n_feat)
test_metrics = metrics(y_test, y_test_pred, len(y_test), n_feat)

results = pd.DataFrame([train_metrics, test_metrics], index=["Train", "Test"])
print("\n=== EVALUATION METRICS ===")
print(results.round(4))
results.round(4).to_csv(os.path.join(BASE_DIR, "task1_metrics.csv"))
coef_table.round(4).to_csv(os.path.join(BASE_DIR, "task1_coefficients.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for a, (yt, yp, title, color) in zip(
    axes,
    [(y_train, y_train_pred, "Training set", "#2c6e91"),
     (y_test, y_test_pred, "Test set", "#c0562d")],
):
    a.scatter(yt, yp, alpha=0.6, color=color, edgecolor="k", linewidth=0.3)
    lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    a.plot(lims, lims, "k--", linewidth=1)
    a.set_xlabel("Actual Selling Price")
    a.set_ylabel("Predicted Selling Price")
    a.set_title(f"{title}: Actual vs Predicted")
plt.tight_layout()
plt.savefig(FIG + "07_actual_vs_predicted.png")
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for a, (yt, yp, title, color) in zip(
    axes,
    [(y_train, y_train_pred, "Training residuals", "#2c6e91"),
     (y_test, y_test_pred, "Test residuals", "#c0562d")],
):
    resid = yt - yp
    a.scatter(yp, resid, alpha=0.6, color=color, edgecolor="k", linewidth=0.3)
    a.axhline(0, color="k", linestyle="--", linewidth=1)
    a.set_xlabel("Predicted Selling Price")
    a.set_ylabel("Residual (Actual - Predicted)")
    a.set_title(title)
plt.tight_layout()
plt.savefig(FIG + "08_residual_plots.png")
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(y_train - y_train_pred, kde=True, ax=axes[0], color="#2c6e91")
axes[0].set_title("Training residual distribution")
sns.histplot(y_test - y_test_pred, kde=True, ax=axes[1], color="#c0562d")
axes[1].set_title("Test residual distribution")
plt.tight_layout()
plt.savefig(FIG + "09_residual_distribution.png")
plt.close()

fig, ax = plt.subplots(figsize=(7, 4.5))
metrics_to_plot = ["MAE", "RMSE"]
results[metrics_to_plot].plot(kind="bar", ax=ax, color=["#2c6e91", "#c0562d"])
ax.set_title("MAE & RMSE: Train vs Test")
ax.set_ylabel("Lakhs (INR)")
plt.tight_layout()
plt.savefig(FIG + "10_metric_comparison.png")
plt.close()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.bar(results.index, results["R2"], color=["#2c6e91", "#c0562d"])
ax.set_ylim(0, 1)
ax.set_ylabel("R\u00b2 score")
ax.set_title("R\u00b2: Train vs Test")
for i, v in enumerate(results["R2"]):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.savefig(FIG + "11_r2_comparison.png")
plt.close()

print("\nAll figures saved to", FIG)
print("DONE")

Raw shape: (451, 9)
  Car_Name  Year  Selling_Price  Present_Price  Kms_Driven Fuel_Type  \
0     ritz  2014           3.35           5.59       27000    Petrol   
1      sx4  2013           4.75           9.54       43000    Diesel   
2     ciaz  2017           7.25           9.85        6900    Petrol   
3  wagon r  2011           2.85           4.15        5200    Petrol   
4    swift  2014           4.60           6.87       42450    Diesel   

  Seller_Type Transmission  Owner  
0      Dealer       Manual      0  
1      Dealer       Manual      0  
2      Dealer       Manual      0  
3      Dealer       Manual      0  
4      Dealer       Manual      0  
<class 'pandas.DataFrame'>
RangeIndex: 451 entries, 0 to 450
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       451 non-null    str    
 1   Year           451 non-null    int64  
 2   Selling_Price  451 non-null    float64
 3   Present_Price

In [12]:
!pip install statsmodels

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 2.6 MB/s eta 0:00:05
   ------ --------------------------------- 1.8/11.3 MB 5.2 MB/s eta 0:00:02
   --------- ------------------------------ 2.6/11.3 MB 6.4 MB/s eta 0:00:02
   ----------- ---------------------------- 3.1/11.3 MB 4.8 MB/s eta 0:00:02
   --------------- ------------------------ 4.5/11.3 MB 4.8 MB/s eta 0:00:02
   ------------------ --------------------- 5.2/11.3 MB 4.4 MB/s eta 0:00:02
   ------------------ --------------------- 5.2/11.3 MB 4.4 MB/s eta 0:00:02
   --------------------- ------------------ 6.0/11.3 MB 3.9 MB/s eta 0:00:02
   ------------------------ --------------- 7.1/11.3 MB 3.9 MB/s eta 0:00:02
   -------------------------- ------------- 7.6/11.3 MB 3.9 MB/s eta 0:00:01
   -------------------------- ------------- 7.6/11.3 MB 3.9 MB/s eta 0:00:01
   ----------

In [13]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()
CSV_PATH = os.path.join(BASE_DIR, "power_marketing_customer_electricity_dataset.csv")
FIG = os.path.join(BASE_DIR, "figs_t2") + os.sep
os.makedirs(FIG, exist_ok=True)

df = pd.read_csv(CSV_PATH)
target = "electricity_consumption_kwh"

core_features = ["household_size", "monthly_income_usd", "age"]

print("Shape:", df.shape)
print("Repeated customers:", df['customer_id'].duplicated().sum(), "extra rows over", df['customer_id'].nunique(), "unique customers")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for a, c in zip(axes, core_features):
    sns.regplot(data=df, x=c, y=target, ax=a, scatter_kws={"alpha": 0.15, "s": 10},
                line_kws={"color": "red"})
    r = df[c].corr(df[target])
    a.set_title(f"{c} vs consumption (r={r:.2f})")
plt.tight_layout()
plt.savefig(FIG + "01_linearity_core_features.png")
plt.close()

numeric_predictors = [
    "household_size", "monthly_income_usd", "age", "home_area_sqft",
    "appliance_count", "temperature_c", "humidity_percent", "price_per_kwh",
    "ac_usage_hours", "heater_usage_hours", "lighting_usage_hours",
    "cooking_usage_hours", "entertainment_usage_hours",
    "previous_hour_consumption_kwh", "previous_day_same_hour_consumption_kwh",
    "rolling_24h_avg_consumption_kwh",
]
corr_full = df[numeric_predictors + [target]].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr_full, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            annot_kws={"size": 7})
plt.title("Correlation matrix: candidate numeric predictors + target")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG + "02_full_correlation_heatmap.png")
plt.close()

X_vif = df[numeric_predictors].dropna()
X_vif = (X_vif - X_vif.mean()) / X_vif.std()
vif_data = pd.DataFrame()
vif_data["Feature"] = numeric_predictors
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data.sort_values("VIF", ascending=False)
print("\n=== VIF (all candidate predictors) ===")
print(vif_data.round(2))
vif_data.round(2).to_csv(os.path.join(BASE_DIR, "task2_vif_full.csv"), index=False)

vif_core = pd.DataFrame()
X_core = df[core_features]
X_core = (X_core - X_core.mean()) / X_core.std()
vif_core["Feature"] = core_features
vif_core["VIF"] = [variance_inflation_factor(X_core.values, i) for i in range(X_core.shape[1])]
print("\n=== VIF (core 3 features only) ===")
print(vif_core.round(2))
vif_core.round(2).to_csv(os.path.join(BASE_DIR, "task2_vif_core.csv"), index=False)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=vif_data, x="VIF", y="Feature", hue="Feature", legend=False, palette="rocket")
ax.axvline(5, color="k", linestyle="--", label="VIF=5 (concern)")
ax.axvline(10, color="red", linestyle="--", label="VIF=10 (severe)")
ax.legend()
ax.set_title("Variance Inflation Factor per candidate predictor")
plt.tight_layout()
plt.savefig(FIG + "03_vif_barplot.png")
plt.close()

df_sorted = df.sort_values(["customer_id", "timestamp"])
model_core = LinearRegression().fit(df[core_features], df[target])
resid_core = df[target].values - model_core.predict(df[core_features])

df_sorted_idx = df_sorted.index
resid_sorted = pd.Series(df[target].values - model_core.predict(df[core_features]),
                          index=df.index).loc[df_sorted_idx]
dw_stat = durbin_watson(resid_sorted.values)
print(f"\nDurbin-Watson statistic (residuals ordered by customer & time): {dw_stat:.3f}")
print("(≈2 = no autocorrelation; toward 0 or 4 = strong positive/negative autocorrelation)")

one_cust = df[df["customer_id"] == df["customer_id"].iloc[0]].sort_values("timestamp")
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(len(one_cust)), one_cust[target].values, marker="o")
ax.set_title(f"Repeated readings for a single customer (id={one_cust['customer_id'].iloc[0]}) -- not i.i.d.")
ax.set_xlabel("Reading sequence")
ax.set_ylabel("Electricity consumption (kWh)")
plt.tight_layout()
plt.savefig(FIG + "04_repeated_measures_example.png")
plt.close()

fitted_core = model_core.predict(df[core_features])
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(fitted_core, resid_core, alpha=0.15, s=10, color="#2c6e91")
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Fitted consumption (kWh)")
ax.set_ylabel("Residual")
ax.set_title("Residuals vs Fitted -- core 3-feature model")
plt.tight_layout()
plt.savefig(FIG + "05_homoscedasticity_core.png")
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(resid_core, kde=True, ax=axes[0], color="#2c6e91")
axes[0].set_title("Residual distribution (core model)")
stats.probplot(resid_core, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot of residuals")
plt.tight_layout()
plt.savefig(FIG + "06_normality_residuals.png")
plt.close()

shapiro_stat, shapiro_p = stats.shapiro(resid_core[:5000])
print(f"\nShapiro-Wilk (first 5000 residuals): stat={shapiro_stat:.4f}, p={shapiro_p:.6f}")

from sklearn.metrics import r2_score
r2_core = r2_score(df[target], fitted_core)
print(f"\nR2 of the 3-feature (household_size, income, age) linear model: {r2_core:.4f}")

print("\nAll figures saved to", FIG)
print("DONE")

Shape: (15000, 38)
Repeated customers: 14250 extra rows over 750 unique customers

=== VIF (all candidate predictors) ===
                                   Feature    VIF
5                            temperature_c  10.72
15         rolling_24h_avg_consumption_kwh   7.89
14  previous_day_same_hour_consumption_kwh   7.51
8                           ac_usage_hours   4.99
9                       heater_usage_hours   3.80
6                         humidity_percent   2.16
12               entertainment_usage_hours   1.75
7                            price_per_kwh   1.70
10                    lighting_usage_hours   1.25
13           previous_hour_consumption_kwh   1.23
11                     cooking_usage_hours   1.17
3                           home_area_sqft   1.00
4                          appliance_count   1.00
0                           household_size   1.00
1                       monthly_income_usd   1.00
2                                      age   1.00

=== VIF (core 3 features on